# Code 8.4.2: Pretraining a mini GPT on tiny Shakespeare

Trains the GPT of Code 8.2.1 (saved as `gpt.py`) on tiny Shakespeare with AdamW, warmup plus cosine decay, and gradient clipping, printing training and validation loss every 200 steps and the run's $6PD$ compute and achieved throughput. The embedding table is restricted to the GPT-2 tokens that occur in the corpus. The final model is saved as `mini_gpt.pt` for Section 5. The timing depends on the machine; ours was a shared 8-core CPU.

This listing loads `mini_gpt.pt`, which Code 8.4.2 writes after pretraining the mini GPT.


In [ ]:
import os
import urllib.request

# Tiny Shakespeare, the corpus Code 8.1.2 downloads as input.txt.
_url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
if not os.path.exists("input.txt"):
    urllib.request.urlretrieve(_url, "input.txt")


In [ ]:
from pathlib import Path

# Code 8.2.1, saved as gpt.py so `from gpt import ...` works in this notebook.
_GPT_PY = 'import math\nfrom dataclasses import dataclass\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\n@dataclass\nclass GPTConfig:\n    V: int = 50257          # vocabulary size\n    n_max: int = 1024       # maximum context length\n    N: int = 12             # number of blocks\n    d: int = 768            # model width\n    h: int = 12             # heads\n    dropout: float = 0.0\n\nclass CausalSelfAttention(nn.Module):\n    def __init__(self, cfg):\n        super().__init__()\n        self.qkv = nn.Linear(cfg.d, 3 * cfg.d)          # W_Q, W_K, W_V in one matrix\n        self.proj = nn.Linear(cfg.d, cfg.d)             # W_O\n        self.h, self.dropout = cfg.h, cfg.dropout\n    def forward(self, x):\n        B, n, d = x.shape\n        q, k, v = self.qkv(x).split(d, dim=-1)\n        q, k, v = (t.view(B, n, self.h, d // self.h).transpose(1, 2) for t in (q, k, v))\n        y = F.scaled_dot_product_attention(q, k, v, is_causal=True,   # the causal mask of Section 6.4\n                                           dropout_p=self.dropout if self.training else 0.0)\n        return self.proj(y.transpose(1, 2).reshape(B, n, d))\n\nclass MLP(nn.Module):\n    def __init__(self, cfg):\n        super().__init__()\n        self.fc = nn.Linear(cfg.d, 4 * cfg.d)\n        self.proj = nn.Linear(4 * cfg.d, cfg.d)\n    def forward(self, x):\n        return self.proj(F.gelu(self.fc(x), approximate="tanh"))   # GPT-2 uses the tanh form of GELU\n\nclass Block(nn.Module):                                 # pre-norm decoder block, no cross-attention\n    def __init__(self, cfg):\n        super().__init__()\n        self.ln1, self.attn = nn.LayerNorm(cfg.d), CausalSelfAttention(cfg)\n        self.ln2, self.mlp = nn.LayerNorm(cfg.d), MLP(cfg)\n        self.drop = nn.Dropout(cfg.dropout)\n    def forward(self, x):\n        x = x + self.drop(self.attn(self.ln1(x)))\n        return x + self.drop(self.mlp(self.ln2(x)))\n\nclass GPT(nn.Module):\n    def __init__(self, cfg):\n        super().__init__()\n        self.cfg = cfg\n        self.tok = nn.Embedding(cfg.V, cfg.d)\n        self.pos = nn.Embedding(cfg.n_max, cfg.d)       # learned absolute positions\n        self.drop = nn.Dropout(cfg.dropout)\n        self.blocks = nn.ModuleList(Block(cfg) for _ in range(cfg.N))\n        self.ln_f = nn.LayerNorm(cfg.d)                 # final LayerNorm after the last block\n        self.apply(self._init)\n        for name, p in self.named_parameters():         # residual branches: std / sqrt(2N)\n            if name.endswith("proj.weight"):\n                nn.init.normal_(p, std=0.02 / math.sqrt(2 * cfg.N))\n    def _init(self, m):\n        if isinstance(m, (nn.Linear, nn.Embedding)):\n            nn.init.normal_(m.weight, std=0.02)\n        if isinstance(m, nn.Linear) and m.bias is not None:\n            nn.init.zeros_(m.bias)\n    def forward(self, idx, targets=None):\n        n = idx.size(1)\n        assert n <= self.cfg.n_max\n        x = self.drop(self.tok(idx) + self.pos(torch.arange(n, device=idx.device)))\n        for block in self.blocks:\n            x = block(x)\n        logits = self.ln_f(x) @ self.tok.weight.T       # output projection tied to the embedding\n        if targets is None:\n            return logits\n        return logits, F.cross_entropy(logits.flatten(0, 1), targets.flatten())\n'
Path('gpt.py').write_text(_GPT_PY)


In [ ]:
import math, time
import tiktoken
import torch
from gpt import GPT, GPTConfig

text = open("input.txt", encoding="utf-8").read()       # tiny Shakespeare (Code 8.1.2)
enc = tiktoken.get_encoding("gpt2")
ids = torch.tensor(enc.encode(text))
vocab = ids.unique()                                    # the GPT-2 tokens that occur in this corpus
data = torch.searchsorted(vocab, ids)                   # renumber them 0 .. len(vocab) - 1
split = int(0.9 * len(data))
train_data, val_data = data[:split], data[split:]       # the same split as the bigram baseline

n, B = 128, 16                                          # context length, sequences per batch
def get_batch(src, gen):
    i = torch.randint(len(src) - n - 1, (B,), generator=gen)
    blk = torch.stack([src[j: j + n + 1] for j in i.tolist()])   # blocks of n + 1 tokens
    return blk[:, :-1], blk[:, 1:]

torch.manual_seed(0)
cfg = GPTConfig(V=len(vocab), n_max=n, N=4, d=128, h=4, dropout=0.1)
model = GPT(cfg)
P = sum(p.numel() for p in model.parameters())
print(f"V = {cfg.V:,}; {P / 1e6:.2f}M parameters, {12 * cfg.N * cfg.d**2 / 1e6:.2f}M of them in the blocks")

# AdamW: weight decay on weight matrices only, not on biases, LayerNorm gains, or positions
decay = [p for name, p in model.named_parameters() if p.dim() == 2 and "pos" not in name]
no_decay = [p for name, p in model.named_parameters() if p.dim() < 2 or "pos" in name]
opt = torch.optim.AdamW([{"params": decay, "weight_decay": 0.1},
                         {"params": no_decay, "weight_decay": 0.0}],
                        lr=1e-3, betas=(0.9, 0.95))
steps, warmup, min_ratio = 800, 50, 0.1
def lr_lambda(s):                                       # linear warmup, then cosine decay to 10%
    if s < warmup:
        return (s + 1) / warmup
    progress = (s - warmup) / (steps - warmup)
    return min_ratio + (1 - min_ratio) * 0.5 * (1 + math.cos(math.pi * progress))
sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)
use_bf16 = torch.cuda.is_available()                    # BF16 autocast on a GPU; plain FP32 on a CPU

@torch.no_grad()
def evaluate(src, iters=10):
    model.eval()
    gen = torch.Generator().manual_seed(123)            # the same batches at every evaluation
    losses = []
    for _ in range(iters):
        x, y = get_batch(src, gen)
        with torch.autocast("cuda", dtype=torch.bfloat16, enabled=use_bf16):
            losses.append(model(x, y)[1].item())
    model.train()
    return sum(losses) / len(losses)

print(f"step 0: val {evaluate(val_data):.2f}, ln V = {math.log(cfg.V):.2f}")
gen = torch.Generator().manual_seed(0)
train_time = 0.0
for step in range(steps):
    t0 = time.time()
    x, y = get_batch(train_data, gen)
    with torch.autocast("cuda", dtype=torch.bfloat16, enabled=use_bf16):
        _, loss = model(x, y)
    opt.zero_grad(set_to_none=True)
    loss.backward()
    gnorm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)   # returns the norm before clipping
    opt.step(); sched.step()
    train_time += time.time() - t0
    if (step + 1) % 200 == 0:
        print(f"step {step + 1}: train {evaluate(train_data):.2f}, val {evaluate(val_data):.2f}, "
              f"grad norm {gnorm:.2f}, lr {sched.get_last_lr()[0]:.1e}")
D = steps * B * n                                       # training tokens
print(f"{D:,} tokens ({D / len(train_data):.1f} epochs) in {train_time:.0f} s; "
      f"6PD = {6 * P * D:.2e} FLOPs, {6 * P * D / train_time / 1e9:.0f} GFLOP/s achieved")
torch.save({"cfg": cfg, "model": model.state_dict(), "vocab": vocab}, "mini_gpt.pt")   # used in Section 5
# V = 11,706; 2.31M parameters, 0.79M of them in the blocks
# step 0: val 9.35, ln V = 9.37
# step 200: train 5.09, val 5.32, grad norm 0.76, lr 9.1e-04
# step 400: train 4.62, val 5.04, grad norm 0.90, lr 6.0e-04
# step 600: train 4.44, val 4.92, grad norm 0.77, lr 2.5e-04
# step 800: train 4.37, val 4.87, grad norm 0.81, lr 1.0e-04
# 1,638,400 tokens (5.4 epochs) in 186 s; 6PD = 2.27e+13 FLOPs, 122 GFLOP/s achieved
